# Unified data preprocessing

Update the client settings in the configuration cell below. Select `OE` for columns such as `Oe1` and `Oe2`, or `ST` for columns such as `St1` and `St2`. The list of columns to drop is configured separately for each dataset.

In [ ]:
import json
import os
import time

import deepl
import pandas as pd
from deep_translator import GoogleTranslator
from deep_translator.exceptions import TooManyRequests
from dotenv import load_dotenv

In [ ]:
# === Client configuration: update this cell for each dataset ===
input_path = r"C:\Users\SakshiMehta\OneDrive - Denison Consulting\Text Analytics data\Untypical\2026 Company Culture Survey_SurveyData_2026-09-15_0137.xlsx"
company_name = "Untypical 2026"
EU_client = True

# Choose "OE" for Oe1/Oe2/... columns or "ST" for St1/St2/... columns.
open_text_type = "OE"

# Dataset-specific columns that should not be processed.
# Example: columns_to_drop = ["Oe4"]
columns_to_drop = []

In [ ]:
OPEN_TEXT_CONFIG = {
    "OE": {
        "prefix": "Oe",
        "excluded_columns": set(),
    },
    "ST": {
        "prefix": "St",
        "excluded_columns": {"Status", "State", "Store", "store"},
    },
}

open_text_type = str(open_text_type).strip().upper()
if open_text_type not in OPEN_TEXT_CONFIG:
    valid_types = ", ".join(OPEN_TEXT_CONFIG)
    raise ValueError(
        f"Invalid open_text_type {open_text_type!r}. Choose one of: {valid_types}."
    )

mode_config = OPEN_TEXT_CONFIG[open_text_type]
open_text_prefix = mode_config["prefix"]
excluded_open_text_columns = mode_config["excluded_columns"]

df = pd.read_excel(input_path, keep_default_na=False)
output_folder = os.path.dirname(input_path)

print(f"Open-text type: {open_text_type} (Excel prefix: {open_text_prefix})")
print(f"Output folder: {output_folder}")

In [ ]:
def translate_deepl_safely(
    texts,
    deepl_client,
    target_lang="EN-US",
    max_batch_bytes=95 * 1024,
    max_batch_items=40,
):
    """Translate texts in batches below DeepL's request-size limit."""
    all_translations = []
    current_batch = []

    def request_size(batch):
        payload = {"text": batch, "target_lang": target_lang}
        return len(json.dumps(payload, ensure_ascii=False).encode("utf-8"))

    def translate_batch(batch):
        results = deepl_client.translate_text(batch, target_lang=target_lang)
        if not isinstance(results, list):
            results = [results]
        return [result.text for result in results]

    for text in texts:
        text = str(text).strip()
        proposed_batch = current_batch + [text]
        batch_too_large = request_size(proposed_batch) > max_batch_bytes
        too_many_items = len(proposed_batch) > max_batch_items

        if current_batch and (batch_too_large or too_many_items):
            all_translations.extend(translate_batch(current_batch))
            current_batch = [text]
            print(f"Translated {len(all_translations)} of {len(texts)} responses")
        else:
            current_batch = proposed_batch

        if len(current_batch) == 1 and request_size(current_batch) > max_batch_bytes:
            raise ValueError(
                "One response is too large to send to DeepL safely. "
                f"Response begins with: {text[:100]!r}"
            )

    if current_batch:
        all_translations.extend(translate_batch(current_batch))

    return all_translations


def translate_google_safely(
    texts,
    translator,
    batch_size=4,
    pause_seconds=1.2,
    max_retries=5,
):
    """Translate texts with controlled batching and rate-limit retries."""
    all_translations = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        for attempt in range(max_retries):
            try:
                batch_translations = translator.translate_batch(batch)
                if isinstance(batch_translations, str):
                    batch_translations = [batch_translations]
                all_translations.extend(batch_translations)
                break
            except TooManyRequests:
                if attempt == max_retries - 1:
                    raise

                wait_seconds = 5 * (2 ** attempt)
                print(
                    f"Google rate limit reached. Waiting {wait_seconds} "
                    "seconds before retrying..."
                )
                time.sleep(wait_seconds)

        time.sleep(pause_seconds)

    return all_translations

In [ ]:
def is_open_text_column(column, prefix, excluded_columns):
    return (
        isinstance(column, str)
        and column.startswith(prefix)
        and column not in excluded_columns
    )


def preprocess_data(
    df,
    open_text_prefix,
    excluded_columns,
    columns_to_drop=None,
):
    if df.empty:
        raise ValueError("The input workbook contains no rows.")

    cleaned = df.copy()
    cleaned.columns = cleaned.iloc[0]
    cleaned = cleaned.iloc[2:].copy()
    cleaned.reset_index(drop=True, inplace=True)
    cleaned = cleaned.iloc[:, 4:].copy()

    if "Status" not in cleaned.columns:
        raise ValueError("The input workbook does not contain a 'Status' column.")

    status_idx = cleaned.columns.get_loc("Status")
    after_status = cleaned.columns[status_idx + 1:]
    response_columns_after_status = [
        column
        for column in after_status
        if is_open_text_column(column, open_text_prefix, excluded_columns)
    ]

    if not response_columns_after_status:
        raise ValueError(
            f"No open-text columns beginning with {open_text_prefix!r} "
            "were found after the 'Status' column."
        )

    first_response_idx = cleaned.columns.get_loc(response_columns_after_status[0])
    cleaned = cleaned.drop(columns=cleaned.columns[status_idx + 1:first_response_idx])

    requested_drops = list(columns_to_drop or [])
    existing_drops = [column for column in requested_drops if column in cleaned.columns]
    missing_drops = [column for column in requested_drops if column not in cleaned.columns]

    if missing_drops:
        print(
            "Warning: configured columns_to_drop not found: "
            + ", ".join(map(str, missing_drops))
        )

    if existing_drops:
        cleaned = cleaned.drop(columns=existing_drops)

    cleaned = cleaned[cleaned["Status"] == "Submitted - Valid"].reset_index(drop=True)
    cleaned = cleaned.dropna(axis=1, how="all")
    cleaned = cleaned.loc[
        :,
        ~cleaned.apply(lambda column: column.astype(str).str.strip().eq("").all()),
    ]

    return cleaned


df_cleaned = preprocess_data(
    df=df,
    open_text_prefix=open_text_prefix,
    excluded_columns=excluded_open_text_columns,
    columns_to_drop=columns_to_drop,
)

In [ ]:
load_dotenv()

if EU_client:
    deepl_api_key = os.getenv("DEEPL_API_KEY")
    if not deepl_api_key:
        raise ValueError(
            "DEEPL_API_KEY was not found. Add it to the .env file before continuing."
        )
    deepl_client = deepl.DeepLClient(deepl_api_key)
    translator = None
else:
    deepl_client = None
    translator = GoogleTranslator(source="auto", target="en")

In [ ]:
open_text_columns = [
    column
    for column in df_cleaned.columns
    if is_open_text_column(
        column,
        open_text_prefix,
        excluded_open_text_columns,
    )
]

if not open_text_columns:
    raise ValueError(
        f"No processable {open_text_type} columns remain after preprocessing."
    )

print("Open-text columns to process: " + ", ".join(open_text_columns))
open_text_dfs = {}
translator_name = "DeepL" if EU_client else "Google"

for open_text_col in open_text_columns:
    columns_to_keep = [
        column
        for column in df_cleaned.columns
        if column not in open_text_columns or column == open_text_col
    ]
    df_part = df_cleaned[columns_to_keep].copy()

    has_response = (
        df_part[open_text_col].notna()
        & df_part[open_text_col].astype(str).str.strip().ne("")
    )
    df_part = df_part.loc[has_response].copy()
    df_part[open_text_col] = df_part[open_text_col].astype(str).str.strip()
    df_part.insert(0, "Res_ID", range(1, len(df_part) + 1))

    translated_col = f"{open_text_col}_Trans"
    texts_to_translate = df_part[open_text_col].tolist()

    if texts_to_translate:
        if EU_client:
            translations = translate_deepl_safely(
                texts=texts_to_translate,
                deepl_client=deepl_client,
                target_lang="EN-US",
            )
        else:
            translations = translate_google_safely(
                texts=texts_to_translate,
                translator=translator,
            )
        df_part[translated_col] = translations
    else:
        df_part[translated_col] = pd.Series(dtype="object", index=df_part.index)

    translation_is_blank = (
        df_part[translated_col].isna()
        | df_part[translated_col].astype(str).str.strip().eq("")
    )
    df_part.loc[translation_is_blank, translated_col] = df_part.loc[
        translation_is_blank, open_text_col
    ]

    df_part.insert(2, translated_col, df_part.pop(translated_col))
    df_part = df_part.sort_values(
        by=translated_col,
        key=lambda column: column.astype(str).str.len(),
        ascending=True,
    )

    open_text_dfs[open_text_col] = df_part
    output_path = os.path.join(
        output_folder,
        f"{company_name}_{open_text_col}_{translator_name}_Trans.xlsx",
    )
    df_part.to_excel(output_path, index=False)
    print(f"Saved: {output_path}")